In [4]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import pickle 
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from tqdm import tqdm 

In [5]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [6]:
class brain(nn.Module):
    def __init__(self, input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, num_layers=2, num_layers_sleep=2):
        super(brain, self).__init__()

        self.rnn = nn.RNN(input_size+sleep_output_size, hidden_wake_size, num_layers, nonlinearity='relu', batch_first=True)
        self.sleep_rnn = nn.RNN(input_size, hidden_sleep_size, num_layers_sleep, nonlinearity='relu', batch_first=True)
        self.sleep_fc = nn.Linear(hidden_sleep_size, sleep_output_size)
        self.wake_fc = nn.Linear(hidden_wake_size, len(tokens))
        self.sleep_output_size = sleep_output_size

    def forward(self, x, x_=None, hw=None, hs=None, sleep=False):
        # print(x.shape, 'x')
        if sleep:
            if hs == None:
                out, hs = self.sleep_rnn(x_)
            else:
                out, hs = self.sleep_rnn(x_, hs)
            # print(out.shape)
            sleep_out = self.sleep_fc(out)
        else:
            sleep_out = torch.zeros((1,x.size(1),self.sleep_output_size))
            
        # print(x.size())
        x = torch.cat((x,sleep_out), dim=2)
        
        if hw == None:
            out, hw = self.rnn(x)
        else:
            out, hw = self.rnn(x, hw)

        out = self.wake_fc(out[:,-1,:])

        if sleep:
            return out, hw, hs
        else:
            return out, hw

In [7]:
class compressor(nn.Module):
    def __init__(self, input_size, hidden_compressor_size, num_layers=1):
        super(compressor, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_compressor_size, num_layers, nonlinearity='relu', batch_first=True)
        self.compressor_fc = nn.Linear(hidden_compressor_size, 2)

    def forward(self, x, hc=None):
        if hc == None:
            out, hc = self.rnn(x)
        else:
            out, hc = self.rnn(x, hc)

        out = self.compressor_fc(out)
        
        return out, hc

In [8]:
def compute_geodesic(hidden1, hidden2):

    total_layers = len(hidden1)
    w = 0

    for ii in range(total_layers):
        w_ = np.array(dist( hidden1[ii], hidden2[ii], 'cosine'))
        w += w_
           
    return w[0][0]/total_layers

In [9]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=8):
        
        one_hot_encoded = np.zeros((len(data), len(tokens)), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, len(tokens)*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), len(tokens)))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,kk*len(tokens):(kk+1)*len(tokens)] = \
                    one_hot_encoded[ii+jj+kk,:]
                    
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [10]:
class Dataset_converter_compressor(Dataset):
    def __init__(self, data, mask):
        total_sample = len(data)
        self.X = np.zeros((total_sample-2, len(tokens)))
        self.y = np.zeros((total_sample-2, 2))
        for ii in range(total_sample-2):
            token = data[ii]
            self.X[ii, ord(token)-65] = 1 
            self.y[ii,mask[ii]] = 1
            

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [11]:
### initial training ###
total_samples = 10000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 30
hidden_compressor_size = 10
hidden_sleep_size = 30
sleep_output_size = 20
num_layers_wake = 1
num_layers_sleep = 1
output_sleep = len(tokens)
input_size = len(tokens)*working_memory
lr = 4e-4
test_acc = []

reps = 10
position = [[], [], [], []]
acc0 = []
acc1 = []
acc2 = []
acc3 = []

for rep in tqdm(range(reps)):
    data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

    data_set = Dataset_converter(data, working_memory, short_term_memory)
    train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

    data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

    data_set = Dataset_converter(data, working_memory, short_term_memory)
    test_loader = DataLoader(data_set, batch_size=1, shuffle=False)

    network1 = brain(input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, num_layers_wake, num_layers_sleep)

    optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
    criterion = torch.nn.CrossEntropyLoss()

    total = 0
    for X, y in train_loader:
        optimizer.zero_grad()

        if total == 0:
            predicted_y, hidden = network1(X)
        else:
            predicted_y, hidden = network1(X, hw=mem)
        
        # print(predicted_y.shape, y.shape)
        loss = criterion(predicted_y, y)
        loss.backward(retain_graph=True)
        optimizer.step()

        with torch.no_grad():
            mem=hidden.clone()
            total += 1

    ###################################
    pos = 1
    total = 0
    for X, y in test_loader:
        if pos==3:
             pos = 0
            #  print(X)
        else:
             pos += 1

        with torch.no_grad():
            if total == 0:
                predicted_y, hidden = network1(X)
            else:
                predicted_y, hidden = network1(X, hw=hidden)

            true_y = y.argmax(axis=1)
            estimated_y = predicted_y.argmax(axis=1)

            total += 1
            if true_y == estimated_y:
                    position[pos].append(1)
            else:
                position[pos].append(0)
    
    acc0.append(100*np.mean(np.array(position[0])))
    acc1.append(100*np.mean(np.array(position[1])))
    acc2.append(100*np.mean(np.array(position[2])))
    acc3.append(100*np.mean(np.array(position[3])))

    print(acc0[-1], acc1[-1], acc2[-1], acc3[-1])

summary = (acc0, acc1, acc2, acc3)

with open('pickle_files/initial_rnn.pickle', 'wb') as f:
     pickle.dump(summary, f)
            

 10%|█         | 1/10 [00:07<01:11,  7.97s/it]

99.95998399359743 17.2468987595038 50.12 99.48


 20%|██        | 2/10 [00:10<00:39,  4.97s/it]

99.1796718687475 17.98719487795118 50.2 99.74


 30%|███       | 3/10 [00:13<00:27,  4.00s/it]

...

99.6548619447779 17.16186474589836 50.05 99.845


 90%|█████████ | 9/10 [00:31<00:03,  3.03s/it]

99.69321061758036 17.171312969632297 50.03999999999999 99.85777777777778


100%|██████████| 10/10 [00:34<00:00,  3.42s/it]

99.72388955582232 17.218887555022008 49.916 99.844


In [15]:
### initial training ###
total_samples = 10000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 30
hidden_compressor_size = 10
hidden_sleep_size = 30
sleep_output_size = 20
num_layers_wake = 1
num_layers_sleep = 1
output_sleep = len(tokens)
input_size = len(tokens)*working_memory
lr = 4e-4
test_acc = []

reps = 10
position1 = [[], [], [], []]
position2 = [[], [], [], []]

acc0_task1 = []
acc1_task1 = []
acc2_task1 = []
acc3_task1 = []

acc0_task2 = []
acc1_task2 = []
acc2_task2 = []
acc3_task2 = []

for rep in tqdm(range(reps)):
    data = get_sequence(total_samples, n_community, n_members)

    data_set = Dataset_converter(data, working_memory, short_term_memory)
    train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

    data_task1 = get_sequence(total_samples, n_community, n_members)
    data_task2 = get_sequence(total_samples, n_community, n_members, train=False)

    data_task1 = Dataset_converter(data_task1, working_memory, short_term_memory)
    test_loader_task1 = DataLoader(data_task1, batch_size=1, shuffle=False)

    data_task2 = Dataset_converter(data_task2, working_memory, short_term_memory)
    test_loader_task2 = DataLoader(data_task2, batch_size=1, shuffle=False)

    network1 = brain(input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, num_layers_wake, num_layers_sleep)

    optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
    criterion = torch.nn.CrossEntropyLoss()

    total = 0
    for X, y in train_loader:
        optimizer.zero_grad()

        if total == 0:
            predicted_y, hidden = network1(X)
        else:
            predicted_y, hidden = network1(X, hw=mem)
        
        # print(predicted_y.shape, y.shape)
        loss = criterion(predicted_y, y)
        loss.backward(retain_graph=True)
        optimizer.step()

        with torch.no_grad():
            mem=hidden.clone()
            total += 1

    ###################################
    pos = 1
    total = 0
    for (X, y), (X_, y_) in zip(test_loader_task1, test_loader_task2):
        if pos==3:
             pos = 0
            #  print(X)
        else:
             pos += 1

        with torch.no_grad():
            if total == 0:
                predicted_y, hidden = network1(X)
            else:
                predicted_y, hidden = network1(X, hw=hidden)

            true_y = y.argmax(axis=1)
            estimated_y = predicted_y.argmax(axis=1)

            if true_y == estimated_y:
                    position1[pos].append(1)
            else:
                position1[pos].append(0)

            ############################################
            if total == 0:
                predicted_y, hidden_ = network1(X_)
            else:
                predicted_y, hidden_ = network1(X_, hw=hidden_)

            true_y = y_.argmax(axis=1)
            estimated_y = predicted_y.argmax(axis=1)

            if true_y == estimated_y:
                    position2[pos].append(1)
            else:
                position2[pos].append(0)

            total += 1
    
    acc0_task1.append(100*np.mean(np.array(position1[0])))
    acc1_task1.append(100*np.mean(np.array(position1[1])))
    acc2_task1.append(100*np.mean(np.array(position1[2])))
    acc3_task1.append(100*np.mean(np.array(position1[3])))

    acc0_task2.append(100*np.mean(np.array(position2[0])))
    acc1_task2.append(100*np.mean(np.array(position2[1])))
    acc2_task2.append(100*np.mean(np.array(position2[2])))
    acc3_task2.append(100*np.mean(np.array(position2[3])))

    print(acc0_task1[-1], acc1_task1[-1], acc2_task1[-1], acc3_task1[-1], acc0_task2[-1], acc1_task2[-1], acc2_task2[-1], acc3_task2[-1])

summary = (acc0_task1, acc1_task1, acc2_task1, acc3_task1, acc0_task2, acc1_task2, acc2_task2, acc3_task2)

with open('pickle_files/initial_rnn_transfer_learning.pickle', 'wb') as f:
     pickle.dump(summary, f)
            

 10%|█         | 1/10 [00:03<00:32,  3.58s/it]

99.95998399359743 26.09043617446979 51.64 99.96000000000001 0.20008003201280514 0.0 45.12 47.64


 20%|██        | 2/10 [00:07<00:28,  3.53s/it]

99.97999199679872 26.350540216086433 51.76 99.98 14.205682272909165 0.0 47.58 35.76


 30%|███       | 3/10 [00:10<00:24,  3.52s/it]

...

99.97999199679872 25.925370148059223 51.205 98.475 36.87474989995999 0.0 48.435 28.095


 90%|█████████ | 9/10 [00:31<00:03,  3.53s/it]

99.98221510826552 25.98372682406296 51.06222222222222 98.62666666666667 32.96874305277667 0.0 48.16444444444444 27.96


100%|██████████| 10/10 [00:35<00:00,  3.54s/it]

99.98399359743898 26.006402561024412 51.124 98.452 29.739895958383357 0.0 48.38 28.112
